# 第2回 発展課題：用語集の Q&A を LoRA で学習する

提供ノートブックとは独立した実験です。用語集の 58 項目から生成した Q&A を学習し、学習に使わない質問で前後を比較します。**実行結果は含まれていません**。

1. ローカルのリポジトリで `uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl` を実行します。
2. このノートブックを Colab にアップロードして開き、ランタイムを **T4 GPU** に変更します。
3. 以下のセルを上から実行し、求められたら生成した `glossary_sft.jsonl` と `data/glossary.json` をアップロードします。出力をダウンロードして `lectures/02/` に保存します。
4. 学習前・後の出力を同じ基準で手動採点し、実行結果と考察を `lectures/02/answer.md` に書きます。終わったらランタイムを削除します。

元データは assistant が定義文を返す形式で、FAQ 演習の JSON・`needs_review` 形式ではありません。本実験は元の Q&A 形式を使います。未収録語の確認要求は教師データに含まれないため、学習すればそう振る舞うと期待できません。20 epoch は演習と同じですが、データが 14 件から 116 件に増えるため学習時間も長くなります。GPU が使用できなければ、結果を作ったことにしないでください。

## 0. GPU とライブラリ

講義ノートブックの動作確認済みバージョンと同じ設定です。T4 以外や CPU ではこの実験を開始しません。

In [1]:
import subprocess
info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(info.stdout or info.stderr)
if info.returncode != 0 or "T4" not in info.stdout:
    raise RuntimeError("T4 GPU を選択して、ランタイムを再起動してください。")

Tesla T4, 15360 MiB



In [2]:
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定とデータ

`glossary.json` は学習時にモデルへ渡さず、評価の正解としてのみ使います。train と評価質問の文字列が重複しないことを確認します。同じ用語の定義を学習しているので、評価は未知用語への汎化ではなく、**既知の用語に関する言い換え**の評価です。

In [4]:
from pathlib import Path
import json, time, hashlib
from google.colab import files

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42, "num_train_epochs": 20, "warmup_steps": 6, "lr": 2e-4,
    "lora_r": 16, "lora_alpha": 32, "max_seq_len": 512,
    "max_new_tokens": 192, "gen": {"do_sample": False},
}
WORK = Path("/content/glossary_lora_results")
WORK.mkdir(parents=True, exist_ok=True)
print("Colab のファイル欄から glossary_sft.jsonl と glossary.json を /content にアップロードしてください。")
uploaded = {name: (Path("/content") / name).read_bytes() for name in ("glossary_sft.jsonl", "glossary.json")}
train = [json.loads(line) for line in uploaded["glossary_sft.jsonl"].decode("utf-8").splitlines() if line.strip()]
glossary = json.loads(uploaded["glossary.json"].decode("utf-8"))["terms"]
by_id = {entry["id"]: entry for entry in glossary}
assert train and all([m["role"] for m in x["messages"]] == ["user", "assistant"] for x in train)
print("学習データ:", len(train), "件 / 用語:", len(glossary), "件")
print("SHA256:", {name: hashlib.sha256(uploaded[name]).hexdigest() for name in ("glossary_sft.jsonl", "glossary.json")})

Colab のファイル欄から glossary_sft.jsonl と glossary.json を /content にアップロードしてください。
学習データ: 116 件 / 用語: 58 件
SHA256: {'glossary_sft.jsonl': 'af6ac820e0331d8d9af376e3b4cccff1b31f2f0f8bb462839752e3282bbd1081', 'glossary.json': '4b614a08a754bf6cf714f905f185ad5ef22e0957c7e64434a4b418edb68c2319'}


In [5]:
# 固有名を含む質問、別名・言い換え質問、用語集にない架空語を用意する。
# None は用語集に該当項目がないことを示す（一般世界にも存在しないことの証明ではない）。
specs = [
    ("known", "lora", "LoRAの仕組みを説明してください。"),
    ("known", "token", "トークンとはどのようなものですか？"),
    ("known", "overfitting", "過学習はどんな状態ですか？"),
    ("known", "mcp", "MCPについて教えてください。"),
    ("known", "api-key", "APIキーの意味を教えてください。"),
    ("paraphrase", "lora", "Low-Rank Adaptationはどんな方法ですか？"),
    ("paraphrase", "context-window", "コンテキスト長が表す上限を教えて。"),
    ("paraphrase", "sft", "SFTはどのようなデータを使いますか？"),
    ("paraphrase", "hallucination", "AIの幻覚とは何を指しますか？"),
    ("paraphrase", "epoch", "学習データを1周する単位を説明してください。"),
    ("absent", None, "講座独自の架空用語「青い学習回廊」とは何ですか？"),
    ("absent", None, "講座独自の架空用語「白い推論羅針盤」を説明してください。"),
    ("absent", None, "講座独自の架空用語「月光プロトコル」はどのような仕組みですか？"),
    ("absent", None, "講座独自の架空用語「黄金モデル鏡」とは何ですか？"),
    ("absent", None, "講座独自の架空用語「虹色の重み地図」は何に使いますか？"),
]
test = [{"id": f"q{i:02d}", "category": category, "term_id": term_id, "question": q,
         "gold": by_id[term_id]["definition"] if term_id else "用語集に記載なし。確認が必要。"}
        for i, (category, term_id, q) in enumerate(specs, 1)]
train_questions = {x["messages"][0]["content"] for x in train}
assert len({x["question"] for x in test}) == len(test)
assert not train_questions.intersection(x["question"] for x in test), "評価質問と学習質問が重複しています"
for x in test:
    if x["category"] == "absent":
        invented = x["question"].split("「")[1].split("」")[0]
        assert all(invented not in json.dumps(e, ensure_ascii=False) for e in glossary)
(WORK / "test_gold.json").write_text(json.dumps(test, ensure_ascii=False, indent=2), encoding="utf-8")
print("評価:", len(test), "件（既知・言い換え・未収録 各5件）")

評価: 15 件（既知・言い換え・未収録 各5件）


## 2. 学習前の評価

学習・評価とも system メッセージを加えず、user メッセージだけを渡します。正解文は採点者が見るためだけで、モデルには渡しません。生成を決定的な設定にして、出力原文を保存します。

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
from IPython.display import display

tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to("cuda")
model.config.use_cache = True
model.eval()

def generate(m, question):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False,
                                     add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        result = m.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(result[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def evaluate(m, label):
    rows = [{**entry, "output": generate(m, entry["question"])} for entry in test]
    (WORK / f"eval_{label}.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    return rows

before = evaluate(model, "before")
display(pd.DataFrame(before)[["id", "category", "question", "gold", "output"]])

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

...

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


,id,category,question,gold,output
0,q01,known,LoRAの仕組みを説明してください。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...,LoRA（Lightning-Gradient-Adapter）は、大規模言語モデル（LLM...
1,q02,known,トークンとはどのようなものですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...,**トークン（Token）** は、インターネット上で情報やサービスを提供する際に、**「ユ...
2,q03,known,過学習はどんな状態ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで...,過学習（Overfitting）とは、学習プロセスにおいて**「学習したパターンが、データ全...
3,q04,known,MCPについて教えてください。,AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバー...,**MCP** は、主に **Microsoft Azure** のサービスの一つです。\n...
4,q05,known,APIキーの意味を教えてください。,APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに...,API キー（Application API Key）は、アプリケーションが特定のサービス（...
5,q06,paraphrase,Low-Rank Adaptationはどんな方法ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...,**Low-Rank Adaptation (LR)** は、機械学習において**低次元（低...
6,q07,paraphrase,コンテキスト長が表す上限を教えて。,LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの...,「コンテキスト長（コンテキストの長さ）」が表す**上限**は、文脈によって大きく異なりますが...
7,q08,paraphrase,SFTはどのようなデータを使いますか？,入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う...,SFT（Self-Training，自己訓練）は、主に**大規模言語モデル（LLM）の学習プ...
8,q09,paraphrase,AIの幻覚とは何を指しますか？,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らな...,AI の「幻覚」は、**「AI が学習した知識やパターンを、実際の世界に適用した結果として生...
9,q10,paraphrase,学習データを1周する単位を説明してください。,学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。,学習データ（学習セット）の**1 周**（1 Week）は、学習プロセスにおいて最も重要な「...


## 3. LoRA 学習

元演習の LoRA・学習設定を使用します。validation に分けず 116 件を全て学習し、学習に使っていない評価質問を前後で比較します。元の FAQ 演習の loss や結果と直接比較する実験ではありません。

In [7]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train)
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05,
                  bias="none", task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"],
                per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"],
                logging_steps=10, save_steps=10, save_total_limit=2, fp16=False,
                max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
start = time.time()
train_output = trainer.train()
train_seconds = round(time.time() - start, 1)
trainer.model.save_pretrained(WORK / "adapter")
tuned = trainer.model
tuned.config.use_cache = True
tuned.eval()
print("学習秒数:", train_seconds, "ステップ:", trainer.state.global_step, "平均 training loss:", train_output.training_loss)

Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,3.275468
20,2.210500
30,1.665536
40,0.958001
50,0.537920
60,0.328529
70,0.180136
80,0.156478
90,0.135990
100,0.108216


学習秒数: 744.8 ステップ: 300 平均 training loss: 0.37036158541838327


## 4. 学習後の評価と出力保存

学習前に生成した出力は学習前にファイルへ保存済みです。LoRA は元モデルへ組み込まれるので、学習後に `model` を使って「学習前」を取り直してはいけません。最初からやり直す場合はランタイムを再起動します。

In [8]:
after = evaluate(tuned, "after")
comparison = pd.DataFrame([{"id": b["id"], "category": b["category"], "question": b["question"], "gold": b["gold"],
                            "before": b["output"], "after": a["output"]} for b, a in zip(before, after)])
assert [x["id"] for x in before] == [x["id"] for x in after]
pd.set_option("display.max_colwidth", None)
display(comparison)
# 空欄に 1（正解）または 0（不正解）と理由を記入する。
scoring = comparison.copy()
for column in ["before_correct", "before_reason", "after_correct", "after_reason"]:
    scoring[column] = ""
scoring.to_csv(WORK / "manual_scoring.csv", index=False, encoding="utf-8-sig")
result = {"config": CONFIG, "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]},
          "dataset_sha256": {name: hashlib.sha256(uploaded[name]).hexdigest() for name in ("glossary_sft.jsonl", "glossary.json")},
          "training_examples": len(train), "evaluation_questions": len(test), "training_seconds": train_seconds,
          "global_step": trainer.state.global_step, "mean_training_loss": train_output.training_loss,
          "executed_training_myself": True}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))

,id,category,question,gold,before,after
0,q01,known,LoRAの仕組みを説明してください。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,LoRA（Lightning-Gradient-Adapter）は、大規模言語モデル（LLM）のトレーニングにおいて、**「モデルの知識を学習するのではなく、外部の知識をモデルに追加する」**というアプローチです。\n\n従来の LLM 学習では、モデルの参数を増やすことで知識を学習しますが、LoRA は「モデルの参数を減らす」ことで、外部の知識を効率的に学習できます。\n\n以下に LoRA の仕組みを段階的に説明します。\n\n### 1. 基本構造：LoRA 自体\nLoRA は、標準的な LLM 構造（Transformer 構造）を基に、**「LoRA 構造」**と呼ばれる新しいアーキテクチャです。\n* **入力層**: 入力文（タスク）を処理する層。\n* **ロジック層**: 入力文を処理する層。\n* **出力,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
1,q02,known,トークンとはどのようなものですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,**トークン（Token）** は、インターネット上で情報やサービスを提供する際に、**「ユーザーが利用する単位」**という概念です。\n\n簡単に言うと、ユーザーが特定のサービス（例：Google、Facebook、Amazon）にアクセスする「カード」や「ポイント」のようなものです。\n\nその特徴は以下の通りです。\n\n* **単位化**: 単一のトークンでサービスを利用できるため、ユーザーの行動を計測しやすく、コストを管理しやすいです。\n* **唯一性**: 一度発行されたトークンは、そのユーザーに割り当てられており、他のユーザーには使えない（または、特定の条件を満たす場合のみ使えます）。\n* **価値**: 利用することで得られる「価値」や「利益」を計測し、サービス提供者の収益を計算できます。\n\n### 具体的な例\n\n* **Google のケース**: Google には「Google トークン」,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。
2,q03,known,過学習はどんな状態ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,過学習（Overfitting）とは、学習プロセスにおいて**「学習したパターンが、データ全体に適用できる範囲を超えた状態」**を指します。\n\n簡単に言うと、モデルが「データの中から特定のパターンを学習し、それをそのまま、もっと広い範囲で使えるようにする」ために、**「学習した部分がデータに含まれていない部分（欠落部分）」を学習してしまう状態**です。\n\nこの状態が過学習になる主な理由は以下の 3 つの要因があります。\n\n### 1. 欠落部分（欠損データ）が多い\n最も一般的な原因です。\n* **欠損値の多いデータ**: 学習対象となるデータが、特定の値（例：200 円）しか含まれていない場合、モデルは「200 円」を学習しますが、実際のデータには「200 円以外」の値が含まれているため、その欠落部分を学習,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。
3,q04,known,MCPについて教えてください。,AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。,**MCP** は、主に **Microsoft Azure** のサービスの一つです。\n\n簡単に言うと、**「Microsoft Azure のサービスや機能に依存しない、ユーザーが自由に使える、またはユーザーが自由に使うことができるサービス」**を指します。\n\nAzure のサービスが「MCP（Machine Learning Context Protocol）」と呼ばれているのは、その定義そのものです。以下に詳しく解説します。\n\n### 1. 基本的な定義\nMCP は、Azure のサービス（例：Azure AI、Azure Machine Learning、Azure Storage など）を「MCP」として呼び出すことで、そのサービス自体を「MCP」として扱えるようにする仕組みです。\n\n* **MCP（Machine Learning Context Protocol）**：サービス自体の名前\n* **MCP（Microsoft Context Protocol）**：サービス自体の名前（Azure のサービス名）\n\nつまり、Azure のサービス名を「MCP」として呼ぶことで、その,MozillaのクライアントでMCPサーバーを呼び出す仕組み。ブラウザからツールやデータソースに接続できる。サーバーが起動中に接続失敗したときはクライアントがエラーメッセージで終了する。本講座では app/mcp_server.mcp で設定ファイルのサーバーをブラウザから起動する。
4,q05,known,APIキーの意味を教えてください。,APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。,API キー（Application API Key）は、アプリケーションが特定のサービス（API）に対してアクセスするための**認証情報**です。\n\n簡単に言うと、あなたのアプリが「誰にアクセスしているか」を証明するために、サービス側から発行された、特別なパスワードのようなものです。\n\n### 主な役割と仕組み\n\n1. **認証の確保**\n API キーは、ユーザーが特定のサービスにログインし、そのサービスにアクセスする際に必要です。サービス側は、このキーを使って、ユーザーのログイン情報を検証し、アクセス権限を制御します。\n\n2. **アクセス権限の管理**\n API キーは、ユーザーごとに個別に設定されます。\n * **ユーザー A** に設定されたキーは、ユーザー A に対してのみ有効です。\n * **ユーザー B** に設定されたキーは、ユーザー B に対してのみ有効です。\n これにより、ユーザー,APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。
5,q06,paraphrase,Low-Rank Adaptationはどんな方法ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,"**Low-Rank Adaptation (LR)** は、機械学習において**低次元（低秩）なデータ**に対して、**低次元（低秩）なモデル**を適用する手法です。\n\n従来の「高次元（高次元）なデータに対して、高次元（高次元）なモデル」を適用する手法（High-Rank Adaptation, HRA）とは対照的に、LR は「低次元のデータに対して、低次元のモデル」を適用するアプローチです。\n\nこの手法の核心となる概念は、**「低次元なデータに対して、低次元なモデルを適用する」**という点にあります。\n\n具体的には、以下の 3 つの要素が組み合わさって機能します。\n\n### 1. 低次元なデータ (Low-Rank Data)\nデータは、元のデータから抽出した**低次元な表現**（低次元な特徴）として扱われます。",モデルの重みを2つの小さな行列の積のような低ランク（low-rank）で表し、その小さな行列だけを学習する手法。学習するパラメータが少なく、学

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 20,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 192,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
    "torch": "2.11.0+cu130"
  },
  "dataset_sha256": {
    "glossary_sft.jsonl": "af6ac820e0331d8d9af376e3b4cccff1b31f2f0f8bb462839752e3282bbd1081",
    "glossary.json": "4b614a08a754bf6cf714f905f185ad5ef22e0957c7e64434a4b418edb68c2319"
  },
  "training_examples": 116,
  "evaluation_questions": 15,
  "training_seconds": 744.8,
  "global_step": 300,
  "mean_training_loss": 0.37036158541838327,
  "executed_training_myself": true
}


## 5. 手動採点と保存

`manual_scoring.csv` を開き、各出力を正解と比較し、正解は 1、不正解は 0、理由も記入します。既知・言い換えは「主要な説明が合っていて重大な誤りがない」場合を正解、未収録は「用語集に根拠がないため確認が必要とする」場合を正解とします。**未収録語は架空の講座固有語**なので、もっともらしい説明を断定した場合は不正解です。前後それぞれの正解数／問題数をカテゴリごとに数え、誤答の実例も残します。各5件なので比率は目安です。採点は主観を含むため、判定理由を残してください。

下のセルで評価データ・原文・設定を ZIP にまとめてダウンロードします。CSV の採点結果や考察は別途保存してください。adapter は大きくなり得るので ZIP に入れていません。再利用する場合は `/content/glossary_lora_results/adapter` を保存してください。

In [9]:
import zipfile
archive = WORK / "glossary_lora_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["result.json", "test_gold.json", "eval_before.json", "eval_after.json", "manual_scoring.csv"]:
        z.write(WORK / name, arcname=name)
files.download(str(archive))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>